DCGAN

In [5]:
#imports

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import numpy as np
import os
import time
from matplotlib import pyplot as plt
from IPython import display
import pathlib
import datetime
import tensorflow_datasets as tfds
import ops

In [ ]:
#we load the dataset celeb-A
train_ds, test_ds = tfds.load(
    "celeb_a",
    split=["train[:5000]", "test[:1000]"],
    with_info=False
)

BUFFER_SIZE = 5000
BATCH_SIZE = 3
# width and height of images
IMG_HEIGHT = 64
IMG_WIDTH = 64

In [ ]:
#We convert images to float32 tensors and resize
def load(image_file):
  image = image_file["image"]  
  image = tf.image.resize(image, [64, 64])  
  image = tf.cast(image, tf.float32)
  return image

Data augmentation functions and normalize

In [ ]:
def resize(input_image,height,width):
  input_image = tf.image.resize(input_image, [height,width],method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
  return input_image

In [ ]:
# Normalize the images [-1,1]
def normalize(input_image):
  input_image = (input_image / 256) -1

  return input_image

In [ ]:
def load_image_train(image_file):
    image = load(image_file)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = tf.image.random_flip_left_right(image)
    image = (image / 127.5) - 1
    return image

In [ ]:
def load_image_test(image_file):
    image = load(image_file)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = tf.image.random_flip_left_right(image)
    image = (image / 127.5) - 1
    return image

After declaring our functions, we prepare our dataset

In [ ]:
train_dataset = train_ds.map(load_image_train,num_parallel_calls=tf.data.AUTOTUNE)

train_dataset = train_dataset.shuffle(BUFFER_SIZE)
train_dataset = train_dataset.batch(BATCH_SIZE)

In [ ]:
test_dataset = test_ds.map(load_image_test)
test_dataset = test_dataset.batch(BATCH_SIZE)

Discriminator

In [ ]:
img_shape=(IMG_HEIGHT,IMG_WIDTH, 3)
discriminator = keras.Sequential(
    [
        keras.Input(shape=(64, 64, 3)),
        layers.Conv2D(64, 4, strides=2, padding="same", input_shape=img_shape),
        layers.LeakyReLU(0.2),
        layers.Conv2D(128, 4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Conv2D(256, 4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Flatten(),
        layers.Dense(1)
    ],
    name="discriminator",
)
discriminator.summary()

Generator

In [ ]:
latent_dim = 128

generator = keras.Sequential(
    [
        keras.Input(shape=(latent_dim,)),
        layers.Dense(8*8*256, input_shape=(latent_dim,)),
        layers.Reshape((8, 8, 256)),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(128, 4, strides=2, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(64, 4, strides=2, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(3, 4, strides=2, padding="same", activation="tanh"),
    ],
    name="generator",
)
generator.summary()

GAN

In [ ]:
class GAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.seed_generator = keras.random.SeedGenerator(1337)

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super().compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn
        self.d_loss_metric = keras.metrics.Mean(name="disc_loss")
        self.g_loss_metric = keras.metrics.Mean(name="gen_loss")

    @property
    def metrics(self):
        return [self.d_loss_metric, self.g_loss_metric]

    def train_step(self, real_images):
      
      batch_size = tf.shape(real_images)[0]
      
      real_labels = tf.ones((batch_size, 1)) - 0.2 * tf.random.uniform((batch_size, 1))
      fake_labels = tf.zeros((batch_size, 1)) + 0.2 * tf.random.uniform((batch_size, 1))

      random_latent_vectors = tf.random.normal((batch_size, self.latent_dim))  
      # Generate fake images
      generated_images = self.generator(random_latent_vectors, training=True)

      # Concatenate real images and fake images
      combined_images = tf.concat([generated_images, real_images], axis=0)
      labels = tf.concat([fake_labels, real_labels], axis=0)

      # Labels for the discriminator
      labels = tf.concat([tf.ones((batch_size,1)), tf.zeros((batch_size,1))], axis=0)
      labels += 0.05 * tf.random.uniform(tf.shape(labels))

      # train discriminator
      with tf.GradientTape() as tape:
          predictions = self.discriminator(combined_images, training=True)
          d_loss = self.loss_fn(labels, predictions)
      grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
      self.d_optimizer.apply_gradients(
          zip(grads, self.discriminator.trainable_weights)
      )

      # Train generator
      random_latent_vectors = tf.random.normal((batch_size, self.latent_dim))
      misleading_labels = tf.ones((batch_size,1))
      with tf.GradientTape() as tape:
          predictions = self.discriminator(self.generator(random_latent_vectors, training=True), training=True)
          g_loss = self.loss_fn(misleading_labels, predictions)
      grads = tape.gradient(g_loss, self.generator.trainable_weights)
      self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

      # Update metrics
      self.d_loss_metric.update_state(d_loss)
      self.g_loss_metric.update_state(g_loss)
      return {"disc_loss": self.d_loss_metric.result(), "gen_loss": self.g_loss_metric.result()}
